<a href="https://colab.research.google.com/github/alexandrebarbosa-dev/projetos-aponti/blob/main/04-prf-2025-python/notebooks/prf_2025_python_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Módulo 4 — Preparação dos Dados com Python

##Aluno: Alexandre Barbosa e Julio Cesar dos Santos

###Objetivos da atividade:

#### Compreensão dos Dados
Revisitar e validar o entendimento inicial da base PRF.

#### Preparação dos Dados
Preparar a base de dados para EDA, dashboard e árvore de decisão.

### Importar bibliotecas

In [1]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

from datetime import datetime
import unicodedata

pd.set_option("display.max_columns", 120)

pd.set_option("display.width", 160)

### Organização da Estrutura de Pastas

In [2]:
# Definindo a pasta raiz
RAIZ = Path.cwd()

PASTAS = [
    "dados_brutos",
    "dados_tratados",
    "notebooks",
    "sql",
    "dashboards",
    "relatorios",
    "apresentacao",
    "logs"
]

for pasta in PASTAS:
    (RAIZ / pasta).mkdir(parents=True, exist_ok=True)

print("Pastas verificadas/criadas:")
for pasta in PASTAS:
    print("-", RAIZ / pasta)

Pastas verificadas/criadas:
- /content/dados_brutos
- /content/dados_tratados
- /content/notebooks
- /content/sql
- /content/dashboards
- /content/relatorios
- /content/apresentacao
- /content/logs


### Definir Parâmetros do Projeto

### Definição de Parâmetros do Projeto
Aqui, definimos os caminhos dos arquivos e as configurações de codificação e separadores, garantindo que os caminhos dos arquivos estejam corretos em relação à sua estrutura de pastas no Google Drive.

In [3]:
# Definição dos caminhos dos arquivos, combinando com a RAIZ do projeto no Google Drive
ARQUIVO_BRUTO = RAIZ / "dados_brutos/acidentes2025.csv"
ARQUIVO_BASE_ANALITICA = RAIZ / "dados_tratados/base_analitica_prf_2025.csv"
ARQUIVO_BASE_MODELAVEL = RAIZ / "dados_tratados/base_modelavel_prf_2025.csv"
ARQUIVO_DICIONARIO = RAIZ / "dados_tratados/dicionario_variaveis_modulo4.csv"
ARQUIVO_DECISOES = RAIZ / "logs/decisoes_tratamento_modulo4.md"
ARQUIVO_README = RAIZ / "README.md"

# Definição de separadores e encodings
SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

### Leitura do Arquivo CSV Bruto
Foi criada uma função para ler o arquivo CSV, tentando diferentes encodings, e então usá-la para carregar os dados brutos na memória.

In [4]:
def ler_csv_prf(caminho, sep=";", encodings=("latin1","utf-8","utf-8-sig")):
    ultimo_erro = None
    for enc in encodings:
        try:
            print(f"Tentando encoding={{enc}}...")
            return pd.read_csv(
                caminho, sep=sep,
                encoding=enc, low_memory=False)
        except Exception as erro:
            ultimo_erro = erro
            print(f"Falhou com {{enc}}: {{erro}}")
    raise ultimo_erro

# Lendo o arquivo CSV bruto usando a função definida
df = ler_csv_prf(ARQUIVO_BRUTO, sep=SEPARADOR)

# Exibindo as primeiras linhas do DataFrame para verificar o carregamento
display(df.head())

Tentando encoding={enc}...
Falhou com {enc}: {erro}
Tentando encoding={enc}...
Falhou com {enc}: {erro}
Tentando encoding={enc}...
Falhou com {enc}: {erro}


FileNotFoundError: [Errno 2] No such file or directory: '/content/dados_brutos/acidentes2025.csv'

### Padronizar nomes das colunas

In [ ]:
def normalizar_nome_coluna(nome):
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("utf-8")
    nome = nome.replace(" ", "_").replace("-", "_").replace("/", "_")
    while "__" in nome:
        nome = nome.replace("__", "_")
    return nome.strip("_")

df.columns = [normalizar_nome_coluna(c) for c in df.columns]
renomear = { "condicao_meteorologica": "condicao_metereologica"}
df = df.rename(columns={
    k:v for k,v in renomear.items() if k in df.columns})

print(df.columns.tolist())

### Conferir Colunas Esperadas

In [ ]:
colunas_esperadas = ["data_inversa", "dia_semana", "horario", "uf", "br", "municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia", "condicao_metereologica", "tipo_pista", "tracado_via", "uso_solo", "pessoas", "mortos", "feridos_leves", "feridos_graves", "feridos", "veiculos"]
faltantes = [c for c in colunas_esperadas if c not in df.columns]
print("Colunas faltantes:", faltantes)
if faltantes:
    print("Atenção: ajuste nomes ou confirme o dicionário oficial da PRF usado no arquivo.")

### Retrato Inicial da Base

In [ ]:
print("Dimensões:", df.shape)
print("Linhas:", df.shape[0])
print("Colunas:", df.shape[1])
display(df.head())
display(df.sample(5, random_state=42))

### Tipos de Dados e Memória

In [ ]:
df.info(memory_usage="deep")

resumo_tipos = (
df.dtypes.astype(str)
.value_counts()
.rename_axis("tipo")
.reset_index(name="qtd_colunas")

)

display(resumo_tipos)

### Diagnóstico de Valores Ausentes

In [ ]:
nulos = pd.DataFrame({ "qtd_nulos": df.isna().sum(),
"perc_nulos": df.isna().mean() * 100
}).sort_values(
"perc_nulos", ascending=False)
display(nulos[nulos["qtd_nulos"] > 0])

### Diagnóstico e Remoção de Duplicidades

In [ ]:
qtd_duplicadas = df.duplicated().sum()
print("Duplicidades exatas:", qtd_duplicadas)
if qtd_duplicadas > 0:
    df = df.drop_duplicates().copy()
    print("Duplicidades removidas.")
    print("Nova dimensão:", df.shape)

### Cardinalidade das Categorias

In [ ]:
categoricas = df.select_dtypes(include="object").columns
cardinalidade = (
df[categoricas]
.nunique(dropna=True)
.sort_values(ascending=False)
.reset_index()
)
cardinalidade.columns = ["variavel", "qtd_categorias"]
display(cardinalidade.head(30))

### Converter Colunas Numéricas

Observação: Para a coluna 'km', tivemos que substituir vírgulas por pontos antes da conversão para numérico, pois, mais na frente, esses dados seriam identificados como nulos na etapa de padronização dos textos.

In [ ]:
colunas_numericas = [ "br","km","pessoas","mortos","feridos", "feridos_leves","feridos_graves", "ilesos","ignorados","veiculos"]

for coluna in colunas_numericas:
  if coluna in df.columns:
    if coluna == 'km':

      df[coluna] = df[coluna].astype(str).str.replace(',', '.', regex=False)
    df[coluna] = pd.to_numeric( df[coluna], errors="coerce")

print(df[[c for c in colunas_numericas if c in df.columns]].dtypes)

### Tratar Datas e Criar Variáveis Temporais

In [ ]:
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")
df["ano"] = df["data_inversa"].dt.year
df["mes"] = df["data_inversa"].dt.month
df["trimestre"] = df["data_inversa"].dt.quarter
df["dia_semana_num"] = df["data_inversa"].dt.dayofweek
df["fim_de_semana"] = df["dia_semana_num"].isin([5, 6]).astype(int)

In [ ]:
#Verificando se as novas variáveis foram criadas corretamente

print('Verificando as novas variáveis temporais:')
display(df[['data_inversa', 'ano', 'mes', 'trimestre', 'dia_semana_num', 'fim_de_semana']].sample(5, random_state=42))
print('\nTipos de dados das novas variáveis:')
display(df[['data_inversa', 'ano', 'mes', 'trimestre', 'dia_semana_num', 'fim_de_semana']].dtypes)

### Tratar Horário e Criar Turno

In [ ]:
horario_limpo = df["horario"].astype(str).str.strip()
df["hora"] = pd.to_datetime(horario_limpo, format="%H:%M:%S", errors="coerce").dt.hour

def classificar_turno(hora):
    if pd.isna(hora):
        return "IGNORADO"
    if 0 <= hora <= 5:
        return "MADRUGADA"
    if 6 <= hora <= 11:
        return "MANHA"
    if 12 <= hora <= 17:
        return "TARDE"
    return "NOITE"

df["turno"] = df["hora"].apply(classificar_turno)

# Verificando as novas colunas que nós criamos
print("Verificando as novas colunas 'hora' e 'turno':")
display(df[['horario', 'hora', 'turno']].sample(5, random_state=42))
print('\nTipos de dados das novas colunas:')
display(df[['hora', 'turno']].dtypes)
print('\nContagem de valores únicos para a coluna "turno":')
display(df['turno'].value_counts())

### Criar Faixa Horária

In [ ]:
def criar_faixa_horaria(hora):
    if pd.isna(hora):
        return "IGNORADO"
    inicio = int(hora // 3) * 3
    fim = inicio + 2
    return f"{inicio:02d}h-{fim:02d}h"

df["faixa_horaria"] = df["hora"].apply(
    criar_faixa_horaria)

display( df["faixa_horaria"].value_counts(dropna=False).sort_index())

In [ ]:
colunas_texto = df.select_dtypes(include="object").columns
for coluna in colunas_texto:
  df[coluna] = (
  df[coluna]
  .astype("string")
  .str.strip()
  .str.upper()
  )
  df[coluna] = df[coluna].replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "NULL": pd.NA})
print("Colunas textuais padronizadas:", len(colunas_texto))

# Exibindo um diagnóstico de nulos atualizado para verificar as mudanças
print("\nDiagnóstico atualizado de valores ausentes após a padronização:")
nulos_atualizado = pd.DataFrame({
    "qtd_nulos": df.isna().sum(),
    "perc_nulos": df.isna().mean() * 100
}).sort_values("perc_nulos", ascending=False)
display(nulos_atualizado[nulos_atualizado["qtd_nulos"] > 0])

### Tratar nulos categóricos

In [ ]:
categoricas_importantes = ["uf", "municipio", "causa_acidente", "tipo_acidente", "fase_dia", "condicao_metereologica", "tipo_pista", "tracado_via", "uso_solo", "classificacao_acidente", "dia_semana"]

for coluna in categoricas_importantes:
    if coluna in df.columns:
            df[coluna] = df[coluna].fillna('IGNORADO')

print("\nVerificação de nulos nas colunas categóricas importantes após o tratamento:")
display(df[categoricas_importantes].isna().sum().sort_values(ascending=False))

### Decisão de Tratamento de Nulos para Variáveis Categóricas

**Estratégia Adotada:**
Para as variáveis categóricas que apresentaram valores ausentes (`NaN` ou `pd.NA`), a decisão foi preencher esses nulos com a string **'IGNORADO'**.

**Justificativa:**
*   **Preservação da Informação:** Evita a remoção de linhas inteiras que poderiam conter informações valiosas em outras colunas, mesmo com a ausência em uma variável categórica específica.
*   **Identificação Explícita:** A categoria 'IGNORADO' torna explícita a condição de dado ausente, o que pode ser útil em análises futuras, permitindo que se investigue se a ausência de informação tem algum padrão ou impacto nos resultados.

### Tratar Nulos Numéricos de Contagem

Observação: os valores nulos das colunas de contagem ("mortos","feridos","feridos_leves", "feridos_graves","pessoas" e "veiculos") foram substituídos por zero, adotando-se a hipótese operacional de que a ausência do valor representa inexistência da ocorrência registrada para aquela categoria de vítima.

In [ ]:
contagem_vitimas = ["mortos","feridos","feridos_leves", "feridos_graves","pessoas","veiculos"]

for coluna in contagem_vitimas:
    if coluna in df.columns:
        df[coluna] = df[coluna].fillna(0)
        print(f"Nulos na coluna '{coluna}' preenchidos com 0.")

print("\nVerificação de nulos nas colunas de contagem após o tratamento:")
display(df[contagem_vitimas].isna().sum().sort_values(ascending=False))

### Criar Variável-Alvo: acidente_fatal

Regra

1 quando mortos >= 1

0 quando mortos = 0

In [ ]:
df["acidente_fatal"] = (df["mortos"] >= 1).astype(int)

#Validação

validacao_alvo = (
    df["acidente_fatal"].value_counts(dropna=False).rename_axis("acidente_fatal").reset_index(name="qtd")
)
validacao_alvo["perc"] = ( validacao_alvo["qtd"] / validacao_alvo["qtd"].sum() * 100)
display(validacao_alvo)

### Validar Logicamente o Alvo

In [ ]:
violacoes = df.loc[
    ((df["mortos"] >= 1) & (df["acidente_fatal"] != 1)) |
    ((df["mortos"] == 0) & (df["acidente_fatal"] != 0))
]

print(f"Violações da regra do alvo: {len(violacoes)}")

if len(violacoes) > 0:
    display(violacoes)

assert len(violacoes) == 0, \
    "Há erro na criação de acidente_fatal."

### Criar Indicadores de Gravidade

acidente_grave:

1 se mortos >= 1 ou feridos_graves >= 1

total_vitimas:

mortos + feridos_leves + feridos_graves

indice_gravidade:

mortos×3 + feridos_graves×2 + feridos_leves

In [ ]:
df["acidente_grave"] = np.where(
    (df["mortos"] >= 1) | (df["feridos_graves"] >= 1),
    1,
    0
)

df["total_vitimas"] = (
    df["mortos"] + df["feridos_leves"] + df["feridos_graves"]
)

df["indice_gravidade"] = (
    df["mortos"] * 3 + df["feridos_graves"] * 2 + df["feridos_leves"]
)

display(df[["mortos", "feridos_leves", "feridos_graves", "total_vitimas",
"indice_gravidade", "acidente_grave"]].head())

### Criar BR Formatada e Chave de Localidade


In [ ]:
def formatar_br(valor):
  if pd.isna(valor) or valor == 0:
    return "BR-IGNORADA"
  return f"BR-{int(valor):03d}"

df["br_formatada"] = df["br"].apply(formatar_br)

df["chave_localidade"] =  (
    df["uf"].astype(str)
    + "_"
    + df["municipio"].astype(str)
    + "_"
    + df["br_formatada"].astype(str)
)

display(df[["br", "br_formatada", "uf", "municipio", "chave_localidade"]].head())

### Checagens Rápidas Após Transformação

Para evitar erros até esse ponto do projeto. Aqui, além do que foi sugerido pela apostila, pensei em criar uma função para fazer essas checagens sempre que precisasse.

Observação: na apostila, ele não pede "Totais de mortos e feridos", mas no pdf do professor esse item está incluído. Para trazer a informação completa, optei por incluir esse item também.


In [ ]:
def fazer_checagens(df):
    return {
        "linhas": len(df),
        "colunas": df.shape[1],
        "acidentes_fatais": int(df["acidente_fatal"].sum()),
        "taxa_fatalidade": float(df["acidente_fatal"].mean()),
        "total_mortos": int(df["mortos"].sum()),
        "total_feridos": int(df["feridos"].sum()) if "feridos" in df.columns else None,
}

checagens = fazer_checagens(df)
checagens


### Ranking Rápido de Categorias

Observação: como adicional, criei uma coluna ranking com uma numeração adequada (começando do 1), em substituição à coluna original, que começava do 0 e poderia confundir na visualização.



In [ ]:
def ranking_categoria(base, coluna, n=10):
    resultado = (
      base[coluna].value_counts(dropna=False).head(n).rename_axis(coluna).reset_index(name="qtd")

  )

    resultado.insert(0, "ranking", range(1, len(resultado) + 1))

    return resultado

display(ranking_categoria(df, "tipo_acidente", 10)
.style.hide(axis="index"))
display(ranking_categoria(df, "causa_acidente", 10)
.style.hide(axis="index"))

### Taxa Fatal por Categoria

In [ ]:
def taxa_fatal_categoria(base, coluna, minimo=30):
  resultado = (
      base.groupby(coluna, dropna=False)
      .agg(
          qtd_acidentes = ("acidente_fatal", "size"),
          acidentes_fatais = ("acidente_fatal", "sum"),
          taxa_fatalidade = ("acidente_fatal", "mean")
      )
      .reset_index()
  )
  resultado = resultado[resultado["qtd_acidentes"] >= minimo]
  return resultado.sort_values("taxa_fatalidade", ascending=False)

display(taxa_fatal_categoria(df, "tipo_acidente", 30)
.head(10)
.style.hide(axis="index")
)

###Gráfico Simples de Conferência


In [ ]:
ax = df["acidente_fatal"].value_counts().sort_index().plot(kind="bar")
ax.set_title("Distribuição da variável alvo acidente_fatal")
ax.set_xlabel("acidente_fatal")
ax.set_ylabel("Quantidade de acidentes")
ax.bar_label(ax.containers[0])
plt.show()


###Gerar Base Analítica Completa

In [ ]:
base_analitica = df.copy()
display(base_analitica.head())

In [ ]:
print("Base analítica:", base_analitica.shape)
print("Colunas:", base_analitica.columns.tolist())

###Selecionar Variáveis Modeláveis


In [ ]:
variaveis_modelaveis = [
"uf", "br_formatada", "municipio", "mes", "trimestre",
"dia_semana", "dia_semana_num", "fim_de_semana",
"hora", "faixa_horaria", "turno", "fase_dia",
"causa_acidente", "tipo_acidente", "condicao_metereologica",
"tipo_pista", "tracado_via", "uso_solo",
"acidente_fatal"
]

variaveis_modelaveis = [c for c in variaveis_modelaveis if c in df.columns]

base_modelavel = df[variaveis_modelaveis].copy()

print("Base modelável:", base_modelavel.shape)

###Verificar data leakage

In [ ]:
variaveis_proibidas = [
    "mortos",
    "feridos",
    "feridos_leves",
    "feridos_graves",
    "total_vitimas",
    "indice_gravidade",
    "acidente_grave",
    "classificacao_acidente"
]

def verificar_data_leakage(base, proibidas):
  presentes = [
      c for c in proibidas
      if c in base.columns
  ]

  if presentes:
    raise ValueError(
        f"Data leakage: {presentes}"
    )
  return "Ok, nenhuma variável proibida foi encontrada."

verificar_data_leakage(
    base_modelavel,
    variaveis_proibidas
)

###Tratamento Final de Nulos na Base Modelável


In [ ]:
for coluna in base_modelavel.columns:
  if coluna == "acidente_fatal":
    continue

  if (
      base_modelavel[coluna].dtype =="object"
      or str(base_modelavel[coluna].dtype) == "string"

  ):
      base_modelavel[coluna] = (
          base_modelavel[coluna].fillna("IGNORADO")
    )
  else:

    base_modelavel[coluna] = (
        base_modelavel[coluna].fillna(-1)
    )

print(
    base_modelavel.isna().sum()
    .sort_values(ascending=False)
    .head()
  )


### Exportar Bases Tratadas


In [ ]:
base_analitica.to_csv(
    ARQUIVO_BASE_ANALITICA,
    index=False,
    sep=SEPARADOR,
    encoding=ENCODING_SAIDA
)

base_modelavel.to_csv(
    ARQUIVO_BASE_MODELAVEL,
    index=False,
    sep=SEPARADOR,
    encoding=ENCODING_SAIDA
)

print("Bases exportadas com sucesso:")
print(ARQUIVO_BASE_ANALITICA)
print(ARQUIVO_BASE_MODELAVEL)

###Reabrir arquivos exportados

In [ ]:
valid_analitica = pd.read_csv(ARQUIVO_BASE_ANALITICA, sep=SEPARADOR, encoding=ENCODING_SAIDA)
valid_modelavel = pd.read_csv(ARQUIVO_BASE_MODELAVEL, sep=SEPARADOR, encoding=ENCODING_SAIDA)

print("Analítica reaberta:", valid_analitica.shape)
print("Modelável reaberta:", valid_modelavel.shape)

assert len(valid_analitica) == len(base_analitica)
assert len(valid_modelavel) == len(base_modelavel)

###Gerar dicionário das variáveis criadas

In [ ]:
linhas_dic = [
{"variavel": "acidente_fatal", "descricao": "1 se mortos >= 1; 0 se mortos = 0", "uso":
"alvo"},
{"variavel": "total_vitimas", "descricao": "mortos + feridos leves + feridos graves", "uso":
"analise/dashboard"},
{"variavel": "indice_gravidade", "descricao": "mortos*3 + feridos_graves*2 + feridos_leves",
"uso": "analise/dashboard"},
{"variavel": "br_formatada", "descricao": "BR padronizada no formato BR-000", "uso":
"analise/modelagem"},
{"variavel": "chave_localidade", "descricao": "UF + município + BR formatada", "uso":
"analise/dashboard"},
]

dicionario = pd.DataFrame(linhas_dic)
dicionario.to_csv(ARQUIVO_DICIONARIO, index=False, sep=SEPARADOR, encoding=ENCODING_SAIDA)
display(dicionario)

###Registrar decisões de tratamento

In [ ]:
texto_decisoes = f"""

# Decisões de tratamento — Módulo 4
Data de geração: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Principais decisões tomadas

- Nomes de colunas padronizados para minúsculas, sem acentos e com underline.
- Colunas numéricas convertidas com `pd.to_numeric(errors='coerce')`.
- Datas convertidas com `pd.to_datetime(errors='coerce')`.
- Categorias ausentes relevantes preenchidas como IGNORADO.
- Variável-alvo: acidente_fatal = 1 quando mortos >= 1.
- Base modelável exclui variáveis derivadas do desfecho.

## Arquivos gerados
- {ARQUIVO_BASE_ANALITICA}
- {ARQUIVO_BASE_MODELAVEL}
- {ARQUIVO_DICIONARIO}

"""

ARQUIVO_DECISOES.write_text(texto_decisoes, encoding="utf-8")
print(ARQUIVO_DECISOES)

###Criar README do projeto

In [ ]:
readme = f"""
# Projeto PRF 2025 — Preparação dos Dados

## Objetivo
Preparar dados de acidentes da PRF 2025 para EDA, Power BI e árvore de decisão.

## Autores
Alexandre Barbosa e Julio Cesar dos Santos

## Variável-alvo
`acidente_fatal = 1` quando `mortos >= 1`; caso contrário, `acidente_fatal = 0`.

## Bases geradas
`{ARQUIVO_BASE_ANALITICA}`: EDA e Power BI.
`{ARQUIVO_BASE_MODELAVEL}`: modelagem, sem data leakage.
"""

ARQUIVO_README.write_text(
    readme,
    encoding="utf-8"
)

print("README criado:", ARQUIVO_README)

###Resumo final da preparação

In [ ]:
resumo_final = pd.DataFrame([
{"item": "linhas_base_analitica", "valor": len(base_analitica)},
{"item": "colunas_base_analitica", "valor": base_analitica.shape[1]},
{"item": "linhas_base_modelavel", "valor": len(base_modelavel)},
{"item": "colunas_base_modelavel", "valor": base_modelavel.shape[1]},
{"item": "taxa_global_acidente_fatal", "valor": base_modelavel["acidente_fatal"].mean()},
])
display(resumo_final)

##Conexão com o Módulo 5 — EDA

In [ ]:
display(base_analitica.head())

###Liste três perguntas que a base analítica permite responder.

*   Comparativo do número de acidentes entre dias de semana e fins de semana?
*   Quais BRs apresentam as maiores taxas de acidentes fatais por trecho em cada estado?
*   Quais causas, tipos de acidente e condições da via apresentam as maiores taxas de fatalidade na base da PRF 2025?

####Qual é a diferença no número de acidentes entre dias de semana e fins de semana?

In [ ]:

resultado = (
    base_analitica
    .groupby("fim_de_semana")
    .agg(
        qtd_acidentes=("acidente_fatal", "size"),
        acidentes_fatais=("acidente_fatal", "sum"),
        taxa_fatalidade=("acidente_fatal", "mean")
    )
    .reset_index()
)

resultado["percentual_fatais"] = (
    resultado["acidentes_fatais"] / resultado["acidentes_fatais"].sum() * 100
)

resultado["fim_de_semana"] = resultado["fim_de_semana"].map({0: "Dia de semana", 1: "Fim de semana"})

display(resultado)

In [ ]:
ax = resultado.plot(
    x="fim_de_semana",
    y="qtd_acidentes",
    kind="bar",
    legend=False
)

ax.set_title("Acidentes por dia de semana e fim de semana")
ax.set_xlabel("")
ax.set_ylabel("Quantidade de acidentes")

for barra in ax.patches:
    ax.annotate(
        f"{int(barra.get_height()):,}".replace(",", "."),
        (barra.get_x() + barra.get_width() / 2, barra.get_height()),
        ha="center",
        va="bottom"
    )

plt.xticks(rotation=0)
plt.show()